# Diagnosis - Silver Layer (dim_diagnosis)

Ingestion incrémentale depuis Bronze avec déduplication sur `diagnosis_code`. Utilise `foreachBatch` + `MERGE` pour upsert.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG    = "hospital"
SOURCE     = f"{CATALOG}.bronze.diagnosis_raw"
TARGET     = f"{CATALOG}.silver.dim_diagnosis"
QUARANTINE = f"{CATALOG}.silver.quarantine_diagnosis"
CHECKPOINT = f"/Volumes/{CATALOG}/bronze/checkpoints/silver/dim_diagnosis"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET} (
  diagnosis_code STRING NOT NULL,
  diagnosis_desc STRING,
  _source_file   STRING,
  _updated_at    TIMESTAMP
) COMMENT 'Dimension diagnostic (SCD1)'
""")

In [0]:
def upsert_dim_diagnosis(batch_df, batch_id):
    # 1. Nettoyage
    clean = (batch_df
        .withColumn("diagnosis_code", F.upper(F.trim("diagnosis_code")))
        .withColumn("diagnosis_desc", F.trim("diagnosis_desc")))

    # 2. Règle qualité
    is_valid = (F.col("diagnosis_code").isNotNull()
                & (F.col("diagnosis_code") != "")
                & F.col("_rescued_data").isNull())

    # 3. Quarantaine (idempotente)
    (clean.filter(~is_valid)
        .withColumn("_rejected_at", F.current_timestamp())
        .write.mode("append")
        .option("txnAppId", "quarantine_diagnosis")
        .option("txnVersion", batch_id)
        .saveAsTable(QUARANTINE))

    # 4. Dédoublonnage : la version la plus récente par code
    w = Window.partitionBy("diagnosis_code").orderBy(F.col("_ingested_at").desc())
    latest = (clean.filter(is_valid)
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .select("diagnosis_code", "diagnosis_desc", "_source_file")
        .withColumn("_updated_at", F.current_timestamp()))

    # 5. MERGE SCD1 : mise à jour seulement si la description change
    (DeltaTable.forName(spark, TARGET).alias("t")
        .merge(latest.alias("s"), "t.diagnosis_code = s.diagnosis_code")
        .whenMatchedUpdateAll(condition="NOT (t.diagnosis_desc <=> s.diagnosis_desc)")
        .whenNotMatchedInsertAll()
        .execute())

In [0]:
(spark.readStream.table(SOURCE)
    .writeStream
    .foreachBatch(upsert_dim_diagnosis)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

display(spark.table(TARGET).orderBy("diagnosis_code"))